Author: Atanazy Gawrysiak

In [9]:
!pip install pycryptodome

In [10]:
import secrets
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
import hashlib
import os
from math import gcd

# (33) Diffie-Hellman Key Exchange


- Choose a large prime $p$ and generator $g$ (public)
- Each party picks a private key from $[1, p-1]$
- Compute public key
$$public\_alice = g^{private\_alice} \mod p$$
- Exchange public keys
- Compute shared secret
$$shared\_alice = public\_bob^{private\_alice}=(g^{private\_bob})^{private\_alice}=g^{private\_bob \cdot private\_alice}$$

In [11]:
hex_str = "ffffffffffffffffc90fdaa22168c234c4c6628b80dc1cd129024 e088a67cc74020bbea63b139b22514a08798e3404ddef9519b3cd 3a431b302b0a6df25f14374fe1356d6d51c245e485b576625e7ec 6f44c42e9a637ed6b0bff5cb6f406b7edee386bfb5a899fa5ae9f 24117c4b1fe649286651ece45b3dc2007cb8a163bf0598da48361 c55d39a69163fa8fd24cf5f83655d23dca3ad961c62f356208552 bb9ed529077096966d670c354e4abc9804f1746c08ca237327ffffffffffffffff"
p = int(hex_str.replace(" ", "").strip(), 16)
g = 2

private_alice = secrets.randbelow(p-1) + 1          # known only by Alice
private_bob = secrets.randbelow(p-1) + 1            # known only by Bob

public_alice = pow(g, private_alice, p)             # Alice shares this with everyone
public_bob = pow(g, private_bob, p)                 # Bob shares thsi with everyone

shared_alice = pow(public_bob, private_alice, p)    # Alice computes the shared secret
shared_bob = pow(public_alice, private_bob, p)      # Bob computes the shared secret

assert shared_alice == shared_bob

print(f"Alice computed:\n{hex(shared_alice)}")
print(f"Bob computed:\n{hex(shared_bob)}")

Alice computed:
0x517e7e484842a338a25799ea42a8828f79e7efab549fb5d20ee6d19c08aeb119fe67bbed312a5a29b31b9b9c6a18472a9a97fff201115400d4c0150403743e95f90d7038766752168abb7660139459f563b91e1516b5e68af5f6054f2e5901ce068412f7c60bb8d5ad0db72fd2559738202cb9308474a1fd622ff77b9572f27dd82286756e202ae078f525a039ca8174e2fbc5b7b62ff48b67889b3f3c3ad6589575ffbb763925e19daa2eb6e731038af1484dcb82241d09f280de6025d83aaa
Bob computed:
0x517e7e484842a338a25799ea42a8828f79e7efab549fb5d20ee6d19c08aeb119fe67bbed312a5a29b31b9b9c6a18472a9a97fff201115400d4c0150403743e95f90d7038766752168abb7660139459f563b91e1516b5e68af5f6054f2e5901ce068412f7c60bb8d5ad0db72fd2559738202cb9308474a1fd622ff77b9572f27dd82286756e202ae078f525a039ca8174e2fbc5b7b62ff48b67889b3f3c3ad6589575ffbb763925e19daa2eb6e731038af1484dcb82241d09f280de6025d83aaa


# (34) MITM attack on Diffie-Hellman

- Code below is an example of scenario withount MITM
- Alice and Bob perform a Diffie-Hellman key exchange to get a shared secret.  
- The secret is hashed (SHA1) to derive a 16-byte AES key.  
- Messages are encrypted with AES-CBC + PKCS7 padding using a random IV.  
- The IV is appended to the ciphertext for decryption.  
- Both parties can encrypt and decrypt messages using the shared key.

In [12]:
def encrypt_cbc(msg: bytes, key: bytes) -> bytes:
    iv = os.urandom(16)

    cipher = AES.new(key, AES.MODE_CBC, iv)
    ciphertext = cipher.encrypt(pad(msg, 16))

    return ciphertext + iv

def decrypt_cbc(data: bytes, key: bytes) -> bytes:
    iv = data[-16:]
    ciphertext = data[:-16]

    cipher = AES.new(key, AES.MODE_CBC, iv)
    msg = unpad(cipher.decrypt(ciphertext), 16)

    return msg

In [13]:
hex_str = "ffffffffffffffffc90fdaa22168c234c4c6628b80dc1cd129024 e088a67cc74020bbea63b139b22514a08798e3404ddef9519b3cd 3a431b302b0a6df25f14374fe1356d6d51c245e485b576625e7ec 6f44c42e9a637ed6b0bff5cb6f406b7edee386bfb5a899fa5ae9f 24117c4b1fe649286651ece45b3dc2007cb8a163bf0598da48361 c55d39a69163fa8fd24cf5f83655d23dca3ad961c62f356208552 bb9ed529077096966d670c354e4abc9804f1746c08ca237327ffffffffffffffff"
p = int(hex_str.replace(" ", "").strip(), 16)
g = 2

# Alice sends p and g to Bob

private_alice = secrets.randbelow(p-1) + 1          # known only by Alice
private_bob = secrets.randbelow(p-1) + 1            # known only by Bob

public_alice = pow(g, private_alice, p)
public_bob = pow(g, private_bob, p)

# Alice sends public_alice to Bob
# Bob sends public_bob to Alice

shared_alice = pow(public_bob, private_alice, p)    # Alice computes the shared secret
shared_bob = pow(public_alice, private_bob, p)      # Bob computes the shared secret

assert shared_alice == shared_bob

msg = b"THIS IS A TEST MESSAGE"

shared_alice_bytes = shared_alice.to_bytes((shared_alice.bit_length() + 7) // 8, 'big')
shared_bob_bytes   = shared_bob.to_bytes((shared_bob.bit_length() + 7) // 8, 'big')

key_alice = hashlib.sha1(shared_alice_bytes).digest()[:16]
key_bob = hashlib.sha1(shared_bob_bytes).digest()[:16]

alices_ciphertext = encrypt_cbc(msg, key_alice)

# Alice sends alices_ciphertext to Bob

alices_decrypted = decrypt_cbc(alices_ciphertext, key_bob)
print(f"Alice decrypted:\n{alices_decrypted}")

bobs_ciphertext = encrypt_cbc(alices_decrypted, key_bob)

# Bob sends bobs_ciphertext to Alice

bobs_decrypted = decrypt_cbc(bobs_ciphertext, key_alice)
print(f"Bob decrypted:\n{bobs_decrypted}")



Alice decrypted:
b'THIS IS A TEST MESSAGE'
Bob decrypted:
b'THIS IS A TEST MESSAGE'


- Code below is an example of scenario with MITM.
- Alice and Bob attempt to perform a Diffie-Hellman key exchange, but Eve intercepts and modifies the public keys.
- Eve replaces both public keys ($public\_alice, public\_bob$) with the value $p$.
- Alice and Bob compute the shared secret as $p^{private\_key}\mod p$, which always results in 0.
- The shared secret is therefore predictable and equal to 0.
- SHA1(0) is used to derive a 16-byte AES key, which Eve can compute as well.
- Messages are encrypted with AES-CBC + PKCS7 padding using a random IV.
- Eve can decrypt, read, and re-encrypt all messages between Alice and Bob, because she knows shared key.

In [14]:
hex_str = "ffffffffffffffffc90fdaa22168c234c4c6628b80dc1cd129024 e088a67cc74020bbea63b139b22514a08798e3404ddef9519b3cd 3a431b302b0a6df25f14374fe1356d6d51c245e485b576625e7ec 6f44c42e9a637ed6b0bff5cb6f406b7edee386bfb5a899fa5ae9f 24117c4b1fe649286651ece45b3dc2007cb8a163bf0598da48361 c55d39a69163fa8fd24cf5f83655d23dca3ad961c62f356208552 bb9ed529077096966d670c354e4abc9804f1746c08ca237327ffffffffffffffff"
p = int(hex_str.replace(" ", "").strip(), 16)
g = 2

# Alice sends p and g to Bob and Eve

private_alice = secrets.randbelow(p-1) + 1          # known only by Alice
private_bob = secrets.randbelow(p-1) + 1            # known only by Bob

public_alice = pow(g, private_alice, p)
public_bob = pow(g, private_bob, p)

# Alice sends public_alice to Bob, but Eve replaces public_alice with p
# Bob sends public_bob to Alice, but Eve replaces public_bob with p

public_alice = p
public_bob = p

shared_alice = pow(public_bob, private_alice, p)    # Alice computes the shared secret
shared_bob = pow(public_alice, private_bob, p)      # Bob computes the shared secret
shared_eve = 0                                      # Eve knows that the shared secret is 0

msg = b"THIS IS A TEST MESSAGE"

shared_alice_bytes = shared_alice.to_bytes((shared_alice.bit_length() + 7) // 8, 'big')
shared_bob_bytes   = shared_bob.to_bytes((shared_bob.bit_length() + 7) // 8, 'big')
shared_eve_bytes   = shared_eve.to_bytes((shared_eve.bit_length() + 7) // 8, 'big')

key_alice = hashlib.sha1(shared_alice_bytes).digest()[:16]
key_bob = hashlib.sha1(shared_bob_bytes).digest()[:16]
key_eve = hashlib.sha1(shared_eve_bytes).digest()[:16]

alices_ciphertext = encrypt_cbc(msg, key_alice)

# Alice sends alices_ciphertext to Bob, Eve intercepts it

alices_decrypted = decrypt_cbc(alices_ciphertext, key_bob)
print(f"Alice decrypted:\n{alices_decrypted}")

eve_decrypted = decrypt_cbc(alices_ciphertext, key_eve)
print(f"Eve decrypted:\n{eve_decrypted}")

bobs_ciphertext = encrypt_cbc(alices_decrypted, key_bob)

# Bob sends bobs_ciphertext to Alice, Eve intercepts it

bobs_decrypted = decrypt_cbc(bobs_ciphertext, key_alice)
print(f"Bob decrypted:\n{bobs_decrypted}")

eve_decrypted = decrypt_cbc(bobs_ciphertext, key_eve)
print(f"Eve decrypted:\n{eve_decrypted}")

assert alices_decrypted == bobs_decrypted == eve_decrypted


Alice decrypted:
b'THIS IS A TEST MESSAGE'
Eve decrypted:
b'THIS IS A TEST MESSAGE'
Bob decrypted:
b'THIS IS A TEST MESSAGE'
Eve decrypted:
b'THIS IS A TEST MESSAGE'


# (41) Unpadded message recovery oracle

This code demonstrates a chosen-ciphertext attack on RSA without padding

1. **Setup RSA keys**

   * $p$ and $q$ are hardcoded primes.
   * $N = pq$
   * $phi = (p-1)*(q-1)$
   * $e$ is hardcoded $65537$, $d$ is computed as modular inverse of $e \mod phi$.

2. **Alice encrypts her message**

   * $alices\_msg$ is encrypted with the public key.

3. **Oracle decryption**

   * The oracle can decrypt any ciphertext using the private key.

4. **Attacker forges a ciphertext**

   * Chooses a random $s$ in $[1, N-1]$.
   * Computes $forged\_ciphertext = (alices\_ciphertext\cdot s^e) \mod N$.

5. **Oracle decrypts forged ciphertext**

   * Returns $forged\_plaintext = sm \mod N$.

6. **Recover original message**

   * Attacker computes $guessed\_plaintext = (forged\_plaintext \cdot s^{-1}) \mod N$.
   * This recovers Alice's original message without directly decrypting her ciphertext.

It all works, because RSA is multiplicatively homomorphic, which means, that:

$$E(m_1)E(m_2) \mod N = E(m_1m_2 \mod N)$$


In [15]:
def encrypt_rsa(m, public_key):
    N, e = public_key
    return pow(m, e, N)

def decrypt_rsa(c, private_key):
    N, d = private_key
    return pow(c, d, N)

In [16]:
p = 1000000007
q = 1000000009
e = 65537
N = p * q
phi = (p - 1) * (q - 1)
d = pow(e, -1, phi)

public_key = (N, e)
private_key = (N, d)

alices_msg = 123456789

# Alice encrypts her message
alices_ciphertext = encrypt_rsa(alices_msg, public_key)
print(f"Alice's Ciphertext:\n{alices_ciphertext}")

# Oracle can decrypt every ciphertext, but only once
alices_plaintext = decrypt_rsa(alices_ciphertext, private_key)
print(f"Alice's Plaintext:\n{alices_plaintext}")

# We capture alices_ciphertext
s = secrets.randbelow(N - 1) + 1
forged_ciphertext = (alices_ciphertext * pow(s, e, N)) % N
print(f"Forged ciphertext:\n{forged_ciphertext}")

# We use oracle to decrpyt forged ciphertext
forged_plaintext = decrypt_rsa(forged_ciphertext, private_key)

guessed_plaintext = (forged_plaintext * pow(s, -1, N)) % N

print(f"Forged decrypted:\n{guessed_plaintext}")

assert alices_plaintext == guessed_plaintext
assert alices_ciphertext != forged_ciphertext # certain ciphertext can be encrypted only once


Alice's Ciphertext:
314174733759806643
Alice's Plaintext:
123456789
Forged ciphertext:
767404921592230882
Forged decrypted:
123456789
